In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt

# ======================================
# 원본 ipynb 최대한 유지 통합본
# Jupyter Notebook 에서 실행 권장
# ======================================

# -----------------------------
# 경로 설정
# -----------------------------
content_path = 'imgs/grammatrix/Tuebingen_Neckarfront.jpg'
style_path = 'imgs/grammatrix/vangogh_starry_night.jpg'

# -----------------------------
# 장치 설정
# -----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# -----------------------------
# 이미지 전처리
# -----------------------------
img_size = 512
loader = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor()
])

unloader = transforms.ToPILImage()


def load_image(path):
    image = Image.open(path).convert('RGB')
    image = loader(image).unsqueeze(0)
    return image.to(device, torch.float)

content_img = load_image(content_path)
style_img = load_image(style_path)

# -----------------------------
# 이미지 출력 함수
# -----------------------------
def imshow(tensor, title=None):
    image = tensor.cpu().clone().squeeze(0)
    image = unloader(image)
    plt.figure(figsize=(8,8))
    plt.imshow(image)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

imshow(content_img, 'Content Image')
imshow(style_img, 'Style Image')

# -----------------------------
# pretrained VGG19 불러오기
# -----------------------------
cnn = torchvision.models.vgg19(pretrained=True).features.to(device).eval()

# weight freeze
for param in cnn.parameters():
    param.requires_grad_(False)

# MaxPool -> AvgPool 변경 (원본 강의 흐름 유지)
for i, layer in enumerate(cnn):
    if isinstance(layer, nn.MaxPool2d):
        cnn[i] = nn.AvgPool2d(kernel_size=2, stride=2)

# -----------------------------
# Normalization
# -----------------------------
cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

class Normalization(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.mean = mean.clone().view(-1,1,1)
        self.std = std.clone().view(-1,1,1)

    def forward(self, img):
        return (img - self.mean) / self.std

# -----------------------------
# Loss 정의
# -----------------------------
def gram_matrix(input):
    a, b, c, d = input.size()
    features = input.view(a * b, c * d)
    G = torch.mm(features, features.t())
    return G.div(a * b * c * d)

class ContentLoss(nn.Module):
    def __init__(self, target):
        super().__init__()
        self.target = target.detach()

    def forward(self, input):
        self.loss = nn.functional.mse_loss(input, self.target)
        return input

class StyleLoss(nn.Module):
    def __init__(self, target_feature):
        super().__init__()
        self.target = gram_matrix(target_feature).detach()

    def forward(self, input):
        G = gram_matrix(input)
        self.loss = nn.functional.mse_loss(G, self.target)
        return input

# -----------------------------
# 사용 레이어
# -----------------------------
content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

# -----------------------------
# 모델 생성
# -----------------------------
def get_style_model_and_losses(cnn, norm_mean, norm_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):

    normalization = Normalization(norm_mean, norm_std).to(device)

    content_losses = []
    style_losses = []

    model = nn.Sequential(normalization)

    i = 0
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = f'conv_{i}'
        elif isinstance(layer, nn.ReLU):
            name = f'relu_{i}'
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d) or isinstance(layer, nn.AvgPool2d):
            name = f'pool_{i}'
        elif isinstance(layer, nn.BatchNorm2d):
            name = f'bn_{i}'
        else:
            raise RuntimeError('Unrecognized layer')

        model.add_module(name, layer)

        if name in content_layers:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module(f'content_loss_{i}', content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module(f'style_loss_{i}', style_loss)
            style_losses.append(style_loss)

    for j in range(len(model) - 1, -1, -1):
        if isinstance(model[j], ContentLoss) or isinstance(model[j], StyleLoss):
            break

    model = model[:j+1]
    return model, style_losses, content_losses

# -----------------------------
# 입력 이미지 (원본 느낌 유지용 content clone)
# -----------------------------
input_img = content_img.clone()

# -----------------------------
# 최적화 함수
# -----------------------------
def run_style_transfer(cnn, norm_mean, norm_std,
                       content_img, style_img, input_img,
                       num_steps=300,
                       style_weight=1000000,
                       content_weight=1):

    model, style_losses, content_losses = get_style_model_and_losses(
        cnn, norm_mean, norm_std, style_img, content_img)

    input_img.requires_grad_(True)
    model.eval()
    model.requires_grad_(False)

    optimizer = optim.LBFGS([input_img])

    run = [0]
    while run[0] <= num_steps:

        def closure():
            input_img.data.clamp_(0,1)
            optimizer.zero_grad()
            model(input_img)

            style_score = 0
            content_score = 0

            for sl in style_losses:
                style_score += sl.loss
            for cl in content_losses:
                content_score += cl.loss

            style_score *= style_weight
            content_score *= content_weight

            loss = style_score + content_score
            loss.backward()

            if run[0] % 50 == 0:
                print(f"step {run[0]}")
                print('Style Loss : {:4f} Content Loss: {:4f}'.format(
                    style_score.item(), content_score.item()))
                print()

                img = input_img.detach().cpu().clone().squeeze(0)
                img = transforms.ToPILImage()(img)
                plt.figure(figsize=(7,7))
                plt.imshow(img)
                plt.title(f'Step {run[0]}')
                plt.axis('off')
                plt.show()

                # 기존 로그
                print(f"step {run[0]}")
                print('Style Loss : {:4f} Content Loss: {:4f}'.format(
                    style_score.item(), content_score.item()))
                print()

            run[0] += 1
            return loss

        optimizer.step(closure)

    input_img.data.clamp_(0,1)
    return input_img

# -----------------------------
# 실행
# -----------------------------
output = run_style_transfer(
    cnn,
    cnn_normalization_mean,
    cnn_normalization_std,
    content_img,
    style_img,
    input_img,
    num_steps=300
)

# -----------------------------
# 결과 출력
# -----------------------------
imshow(output, title='Output Image')


device: cuda


FileNotFoundError: [Errno 2] No such file or directory: 'content.jpg'